# Lab 01 · Prompting and small evaluations

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## Give the model criteria, then test them
**So far:** Lab 00 compared an unguided model with a readiness screen.

**New:** a guided prompt, a JSON contract and a five-case evaluation. Review slide 21: `ops_ready` means automated CI/CD, monitoring, rapid provisioning and teams owning operations. Our simplified screen considers microservices only when `ops_ready` and `independent_deploys` are both true. It remains a screen, not a proof of architecture quality.

The last step uses a tiny event-delivery example to explain tokens, checkable prompting and reasoning-model budgets.

This screen addresses the stated independent-release goal; it does not exclude the independent-scaling motivations in the lecture.

**At a glance**

![Lab 01 at a glance: Labelled case (5 readiness tests), then Guided prompt (checklist + example), then Model (JSON reply), then Parse (parse_reply()), then Score (pass · fail · invalid), then ≥ 80%.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/01.svg)

## Review your microservices lectures

- **Microservices & Their Patterns**, slides 19, 20, 21.
- **Microservices Lecture 2 - Condensed DDD**, slides 51, 52.

**Recall:** Operational prerequisites; At-least-once delivery and event identity; Criteria-based architecture decisions.

Before running the example:

1. Why does CI/CD without monitoring fail the readiness screen?
2. Why do three deliveries not necessarily represent three distinct business events?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

Prompting supplies missing domain context. Evaluation separates malformed replies from wrong labels and tests representative combinations.

A short justification is verifiable evidence, not access to private model reasoning.

| Key term | Meaning |
| --- | --- |
| **Few-shot prompting** | Including one or more worked examples in the prompt, so the model sees what a good answer looks like. |
| **Structured output** | A reply in a fixed format, such as JSON, that code can parse. |
| **Evaluation set** | Labelled cases with known answers, used to score a prompt or a system. |
| **Quality gate** | A check that stops the run when the score falls below a threshold. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Include criteria, one example and the output contract
The worked example isolates a common mistake: readiness without an independent-release need. Ask for a brief justification that a student can check against the supplied facts.

In [ ]:
import json
POLICY = ("Consider microservices only if ops_ready AND independent_deploys are true; otherwise modular monolith. "
          "ops_ready means CI/CD, monitoring, rapid provisioning and teams owning operations are all ready.")
WORKED_EXAMPLE = ('Case: {"ops_ready": true, "independent_deploys": false}\n'
                  'Reply: {"decision": "modular monolith", "reason": "A shared release meets the need."}')

def guided_prompt(case):
    return (f"Screen: {POLICY}\nExample:\n{WORKED_EXAMPLE}\nCase: {json.dumps(case)}\n"
            'Check both facts. Reply with JSON only: {"decision": "...", "reason": "one sentence"}')

print(guided_prompt({"ops_ready": False, "independent_deploys": True}))

## Step 2 · Reject unusable output
A reply must be a JSON object with a known decision and a non-empty reason. Valid JSON such as `[]` is still the wrong schema.

In [ ]:
LABELS = ("modular monolith", "microservices")
def parse_reply(reply):
    try:
        data = json.loads(reply)
    except json.JSONDecodeError:
        raise ValueError("not JSON") from None
    if not isinstance(data, dict) or not isinstance(data.get("reason"), str) or not data["reason"].strip():
        raise ValueError("expected an object with a non-empty reason")
    decision = str(data.get("decision", "")).strip().lower()
    if decision not in LABELS:
        raise ValueError(f"unknown decision {decision!r}")
    return decision

for reply in ['{"decision":"modular monolith","reason":"Missing prerequisites."}', "[]", "Microservices!"]:
    try:
        print("accepted:", parse_reply(reply))
    except ValueError as error:
        print("rejected:", error)

## Step 3 · Cover both facts and a realistic missing-prerequisite case
The last case has pipelines but no monitoring: it must still set `ops_ready=False`. A numerical team-size threshold is not a decision rule in these slides.

In [ ]:
tests = [
    ({"ops_ready": False, "independent_deploys": False}, "modular monolith", "neither fact"),
    ({"ops_ready": True, "independent_deploys": True}, "microservices", "both facts"),
    ({"ops_ready": True, "independent_deploys": False}, "modular monolith", "readiness only"),
    ({"ops_ready": False, "independent_deploys": True}, "modular monolith", "release need only"),
    ({"ops_ready": False, "independent_deploys": True, "note": "CI/CD exists; monitoring is missing"},
     "modular monolith", "one prerequisite missing"),
]
scripted = ["modular monolith", "microservices", "microservices", "modular monolith", "modular monolith"]

## Step 4 · Score the screen
Count a wrong label separately from an invalid reply. The 80% gate is a classroom example, not an acceptable production safety threshold. Inspect the reasons as well: matching a label does not establish a good design.

In [ ]:
counts = {"pass": 0, "fail": 0, "invalid": 0}
for (case, expected, why), label in zip(tests, scripted):
    example = json.dumps({"decision": label, "reason": "Scripted practice reply."})
    try:
        decision = parse_reply(ask(guided_prompt(case), example=example))
        verdict = "pass" if decision == expected else "fail"
    except ValueError as error:
        decision, verdict = str(error), "invalid"
    counts[verdict] += 1
    print(verdict.upper(), why, "->", decision)
accuracy = counts["pass"] / len(tests)
print(counts, f"accuracy: {accuracy:.0%}")
assert accuracy >= 0.8, "Below the classroom quality gate; inspect failed and invalid replies."

## Step 5 · Tokens, checkable steps and reasoning budgets
An LLM processes tokens (pieces of text), not a database of event identities. Ask it to distinguish *deliveries* from *distinct events*, then check with Python. This reviews at-least-once delivery from DDD slide 52.

A decomposition prompt asks for an explicit, short checkable result. It does not expose private internal reasoning, and it does not guarantee correctness. A reasoning model may spend additional output-budget tokens before producing its visible answer; support and usage fields vary by provider. Here both calls use the same configured model, so this is a prompting comparison, **not** an experiment comparing model families. Practice answers are fixtures.

In [ ]:
events = ["event-17", "event-17", "event-18"]  # duplicate delivery of one business event
simple = f"Delivered event ids: {events}. How many distinct business events? Reply with a number."
guided = (f"Delivered event ids: {events}. Deduplicate by eventId, then count. "
          'Reply with JSON only: {"unique_ids": [...], "count": number}.')
print("Simple prompt:", ask(simple, example="3"))
print("Checkable prompt:", ask(guided, example='{"unique_ids":["event-17","event-18"],"count":2}'))
print("Python check:", len(set(events)), "distinct events from", len(events), "deliveries")

## What you should see
The parser rejects a list and non-JSON text. Practice scores 4/5 (80%): the readiness-only fixture is deliberately wrong. The last step contrasts a scripted wrong count of 3 with a checkable count of 2 distinct events.

## Try it
1. Fix the deliberately wrong readiness-only fixture and rerun the evaluation.
2. Remove monitoring but retain the other prerequisites. Explain the correct `ops_ready` value.
3. Add another delivery of `event-18`. How many business events are there now? Why must a consumer deduplicate effects, not merely count messages?
4. In live mode, compare both prompts. Report observed answers rather than assuming the checkable prompt always wins.

## Takeaway
Specify the decision criteria and output contract, test counterexamples, and use deterministic checks where possible. Tokens, generated explanations and reasoning budgets are separate from architectural evidence.